# 28 · Ativos, engajamento e churn por tipo de saída

Pergunta: como era o engajamento de quem saiu, por tipo de saída (`motivo_fim`), olhando para
trás, antes de a saída ser registrada?

- **Tipo de saída**: `motivo_fim` de `assinaturas_limpa` (renovação desligada, falha de pagamento,
  chargeback, reembolso).
- **Engajamento retroativo**: os mesmos níveis de `src/engajamento.py`, com a data de referência de
  cada conta: a véspera da saída (`data_fim` − 1 dia), 30 dias antes da saída e, para quem desligou
  a renovação, a véspera do desligamento.
- **Recorte**: os treinos só existem a partir de 01/01/2026 e o nível "possível churn" precisa de
  45 dias de histórico. Entram as saídas de 17/03 a 30/09/2026, que têm histórico completo nas
  duas datas (véspera e 30 dias antes).
- **Comparação**: assinantes ativos no corte (30/09/2026).

In [1]:
import sys
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento, NIVEIS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
INICIO_TREINOS = pd.Timestamp("2026-01-01")

assin = pd.read_parquet(P / "assinaturas_limpa.parquet")
resumo = pd.read_parquet(P / "assinaturas_resumo_eventos.parquet", columns=["user_id", "primeiro_desligamento", "ultimo_desligamento", "data_carencia"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data"])
a = assin.merge(resumo, on="user_id", how="left", validate="one_to_one")
print(len(assin), "assinaturas -> depois da junção:", len(a), "|", len(ativ), "treinos")

32583 assinaturas -> depois da junção: 32583 | 1252488 treinos


## 1. Quantos saíram, por tipo de saída

In [2]:
enc = a[~a.ativa_no_corte].copy()
print("assinaturas encerradas:", len(enc), "| ativas no corte:", int(a.ativa_no_corte.sum()))
enc["ano_fim"] = enc.data_fim.dt.year
t = pd.crosstab(enc.motivo_fim, enc.ano_fim, margins=True, margins_name="total")
print(t); print()
print("% da coluna"); print(pd.crosstab(enc.motivo_fim, enc.ano_fim, normalize="columns", margins=True, margins_name="total") * 100); print()
print("% por plano (linha) e duração mediana em dias")
g = enc.groupby("motivo_fim").agg(saidas=("user_id", "size"), pct_mensal=("plano_atual", lambda s: (s == "mensal").mean() * 100),
                                   duracao_mediana=("duracao_dias", "median"))
print(g)
print(); print("% da linha por canal de cobrança")
print(pd.crosstab(enc.motivo_fim, enc.canal_cobranca, normalize="index", margins=True, margins_name="total") * 100)

assinaturas encerradas: 17830 | ativas no corte: 14753
ano_fim              2024  2025  2026  total
motivo_fim                                  
chargeback             46   118   128    292
falha_pagamento       160   586   575   1321
reembolso               1    39    32     72
renovacao_desligada  1591  6890  7664  16145
total                1798  7633  8399  17830

% da coluna
ano_fim              2024  2025  2026  total
motivo_fim                                  
chargeback            2.6   1.5   1.5    1.6
falha_pagamento       8.9   7.7   6.8    7.4
reembolso             0.1   0.5   0.4    0.4
renovacao_desligada  88.5  90.3  91.2   90.5

% por plano (linha) e duração mediana em dias
                     saidas  pct_mensal  duracao_mediana
motivo_fim                                              
chargeback              292        69.5             15.0
falha_pagamento        1321        90.9            166.0
reembolso                72         0.0            376.0
renovacao_desli

## 2. Engajamento antes da saída

In [3]:
LIM = INICIO_TREINOS + pd.Timedelta(days=44 + 31)   # 17/03/2026: 45 dias de histórico já 31 dias antes da saída
s = enc[enc.data_fim >= LIM].copy()
print("saídas em 2026:", int((enc.ano_fim == 2026).sum()), "-> de", LIM.date(), "ao corte:", len(s))
print(s.motivo_fim.value_counts().to_dict())
ini = s.set_index("user_id").inicio

def nivel_em(ref, nome):
    e = niveis_engajamento(ativ, ref.index, ref, inicio=ini)
    return e.set_index("user_id").add_suffix("_" + nome)

fim = s.set_index("user_id").data_fim
vespera = nivel_em(fim - pd.Timedelta(days=1), "vespera")
d30 = nivel_em(fim - pd.Timedelta(days=31), "30d_antes")
s = s.join(vespera[["nivel_vespera", "treinos_30d_vespera", "dias_sem_treinar_vespera", "nunca_treinou_vespera"]], on="user_id") \
     .join(d30[["nivel_30d_antes", "treinos_30d_30d_antes"]], on="user_id")
print("depois das junções:", len(s))
# quem assinou há menos de 31 dias da saída não existia como assinante 30 dias antes
s.loc[s.inicio > s.data_fim - pd.Timedelta(days=31), "nivel_30d_antes"] = None
print("sem nível 30 dias antes (assinatura com menos de 31 dias):", int(s.nivel_30d_antes.isna().sum()))

def tabela(d, linha, col):
    n = pd.crosstab(d[linha], d[col], margins=True, margins_name="total").reindex(columns=NIVEIS + ["total"])
    p = (pd.crosstab(d[linha], d[col], normalize="index", margins=True, margins_name="total") * 100).reindex(columns=NIVEIS)
    return n, p

print(); print("== na véspera da saída"); n1, p1 = tabela(s, "motivo_fim", "nivel_vespera"); print(n1); print(p1)
print(); print("== 30 dias antes da saída"); n2, p2 = tabela(s[s.nivel_30d_antes.notna()], "motivo_fim", "nivel_30d_antes"); print(n2); print(p2)
print(); print("treinos nos 30 dias antes da saída (média / mediana) e dias sem treinar na véspera (mediana)")
print(s.groupby("motivo_fim").agg(treinos_media=("treinos_30d_vespera", "mean"), treinos_mediana=("treinos_30d_vespera", "median"),
                                   dias_sem_treinar_mediana=("dias_sem_treinar_vespera", "median"), nunca_treinou=("nunca_treinou_vespera", "sum")))

saídas em 2026: 8399 -> de 2026-03-17 ao corte: 6123
{'renovacao_desligada': 5587, 'falha_pagamento': 423, 'chargeback': 89, 'reembolso': 24}


depois das junções: 6123
sem nível 30 dias antes (assinatura com menos de 31 dias): 442

== na véspera da saída
nivel_vespera        1. alto  2. médio  3. baixo  4. possível churn  total
motivo_fim                                                                
chargeback                75        14         0                  0     89
falha_pagamento          348        59         3                 13    423
reembolso                  2         0         1                 21     24
renovacao_desligada     1443      2723       662                759   5587
total                   1868      2796       666                793   6123
nivel_vespera        1. alto  2. médio  3. baixo  4. possível churn
motivo_fim                                                         
chargeback              84.3      15.7       0.0                0.0
falha_pagamento         82.3      13.9       0.7                3.1
reembolso                8.3       0.0       4.2               87.5
renovacao_desligada    

In [4]:
# Comparação: assinantes ativos no corte, e os ativos com a renovação já desligada
at = a[a.ativa_no_corte]
ea = niveis_engajamento(ativ, at.user_id, CORTE, inicio=at.set_index("user_id").inicio).merge(at[["user_id", "plano_atual", "renovacao_automatica"]], on="user_id")
print("ativos no corte:", len(ea))
ea["grupo"] = ea.renovacao_automatica.map({"sim": "ativo, renovação ligada", "nao": "ativo, renovação desligada"})
na, pa = tabela(ea, "grupo", "nivel"); print(na); print(pa)

ativos no corte: 14753
nivel                       1. alto  2. médio  3. baixo  4. possível churn  total
grupo                                                                            
ativo, renovação desligada      445       857       264                684   2250
ativo, renovação ligada       10244      1514       101                644  12503
total                         10689      2371       365               1328  14753
nivel                       1. alto  2. médio  3. baixo  4. possível churn
grupo                                                                     
ativo, renovação desligada     19.8      38.1      11.7               30.4
ativo, renovação ligada        81.9      12.1       0.8                5.2
total                          72.5      16.1       2.5                9.0


In [5]:
print("== na véspera da saída, por plano (% da linha)")
for pl in ["mensal", "anual"]:
    d = s[s.plano_atual == pl]
    n, p = tabela(d, "motivo_fim", "nivel_vespera")
    print(pl, len(d), "saídas"); print(p.join(n["total"].rename("saídas"))); print()
print("ativos no corte, por plano (% da linha)")
print((pd.crosstab(ea.plano_atual, ea.nivel, normalize="index") * 100).reindex(columns=NIVEIS))
print()
print("== trajetória: nível 30 dias antes × nível na véspera (% da linha), todas as saídas")
d = s[s.nivel_30d_antes.notna()]
print((pd.crosstab(d.nivel_30d_antes, d.nivel_vespera, normalize="index", margins=True, margins_name="total") * 100).reindex(columns=NIVEIS))
print(pd.crosstab(d.nivel_30d_antes, d.nivel_vespera, margins=True, margins_name="total"))

== na véspera da saída, por plano (% da linha)
mensal 4446 saídas
                     1. alto  2. médio  3. baixo  4. possível churn  saídas
motivo_fim                                                                 
chargeback              81.5      18.5       0.0                0.0      65
falha_pagamento         84.0      14.9       0.8                0.3     376
renovacao_desligada     30.1      55.9      11.2                2.8    4005
total                   35.4      51.8      10.1                2.6    4446

anual 1677 saídas
                     1. alto  2. médio  3. baixo  4. possível churn  saídas
motivo_fim                                                                 
chargeback              91.7       8.3       0.0                0.0      24
falha_pagamento         68.1       6.4       0.0               25.5      47
reembolso                8.3       0.0       4.2               87.5      24
renovacao_desligada     14.9      30.7      13.5               40.8    1582
tot

## 3. Quem desligou a renovação: engajamento na véspera do desligamento

A decisão de sair é tomada no desligamento, não no fim do período pago.

In [6]:
r = s[(s.motivo_fim == "renovacao_desligada")].copy()
print("saídas por renovação desligada no recorte:", len(r), "| sem data de desligamento:", int(r.ultimo_desligamento.isna().sum()))
r = r[r.ultimo_desligamento.notna()]
r["data_deslig"] = r.ultimo_desligamento.dt.normalize()
r["dias_deslig_ate_fim"] = (r.data_fim - r.data_deslig).dt.days
print("dias do desligamento ao fim:", r.dias_deslig_ate_fim.describe()[["mean", "25%", "50%", "75%", "max"]].round(0).to_dict())
print(r.groupby("plano_atual").dias_deslig_ate_fim.describe()[["count", "25%", "50%", "75%"]])
ok = r[r.data_deslig - pd.Timedelta(days=1) >= INICIO_TREINOS + pd.Timedelta(days=44)]
print("com 45 dias de histórico antes do desligamento:", len(ok), "de", len(r))
ed = niveis_engajamento(ativ, ok.user_id, ok.set_index("user_id").data_deslig - pd.Timedelta(days=1), inicio=ok.set_index("user_id").inicio)
ok = ok.merge(ed[["user_id", "nivel"]].rename(columns={"nivel": "nivel_deslig"}), on="user_id")
print()
print("nível na véspera do desligamento × plano (% da linha)")
n, p = tabela(ok, "plano_atual", "nivel_deslig"); print(n); print(p)
print()
print("mesmas contas: véspera do desligamento × véspera do fim (% da linha)")
print((pd.crosstab(ok.nivel_deslig, ok.nivel_vespera, normalize="index", margins=True, margins_name="total") * 100).reindex(columns=NIVEIS))

saídas por renovação desligada no recorte: 5587 | sem data de desligamento: 0
dias do desligamento ao fim: {'mean': 55.0, '25%': 10.0, '50%': 19.0, '75%': 42.0, 'max': 363.0}
              count  25%   50%   75%
plano_atual                         
anual       1,582.0 45.0 154.0 247.8
mensal      4,005.0  7.0  15.0  22.0
com 45 dias de histórico antes do desligamento: 4707 de 5587



nível na véspera do desligamento × plano (% da linha)
nivel_deslig  1. alto  2. médio  3. baixo  4. possível churn  total
plano_atual                                                        
anual             255       230        26                191    702
mensal           2280      1595        93                 37   4005
total            2535      1825       119                228   4707
nivel_deslig  1. alto  2. médio  3. baixo  4. possível churn
plano_atual                                                 
anual            36.3      32.8       3.7               27.2
mensal           56.9      39.8       2.3                0.9
total            53.9      38.8       2.5                4.8

mesmas contas: véspera do desligamento × véspera do fim (% da linha)
nivel_vespera      1. alto  2. médio  3. baixo  4. possível churn
nivel_deslig                                                     
1. alto               38.5      55.5       3.6                2.4
2. médio              17.2      

In [7]:
final = n1.join(p1.add_prefix("% ")).rename_axis("grupo")
extra = na.join(pa.add_prefix("% ")).rename_axis("grupo")
extra.index = [i if i != "total" else "ativos no corte (total)" for i in extra.index]
final.index = ["saída: " + i if i != "total" else "saídas de 17/03 a 30/09/2026 (total)" for i in final.index]
final = pd.concat([final, extra])
final.to_csv(S / "tipo_saida_engajamento_antes_da_saida.csv", encoding="utf-8-sig")
print(final)

                                      1. alto  2. médio  3. baixo  4. possível churn  total  % 1. alto  % 2. médio  % 3. baixo  % 4. possível churn
saída: chargeback                          75        14         0                  0     89       84.3        15.7         0.0                  0.0
saída: falha_pagamento                    348        59         3                 13    423       82.3        13.9         0.7                  3.1
saída: reembolso                            2         0         1                 21     24        8.3         0.0         4.2                 87.5
saída: renovacao_desligada               1443      2723       662                759   5587       25.8        48.7        11.8                 13.6
saídas de 17/03 a 30/09/2026 (total)     1868      2796       666                793   6123       30.5        45.7        10.9                 13.0
ativo, renovação desligada                445       857       264                684   2250       19.8        38